In [3]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

print(f'TensorFlow version: {tf.__version__}')
print(f'Pandas version:     {pd.__version__}')
tf.random.set_seed(42)
np.random.seed(42)

ModuleNotFoundError: No module named 'tensorflow.python'

In [ ]:
df_raw = pd.read_csv('dataset_cus_purchase.csv')
print('Shape:', df_raw.shape)
df_raw.head(10)

In [ ]:
print('=== Data Types ===')
print(df_raw.dtypes)
print('\n=== Missing Values ===')
print(df_raw.isnull().sum())
print('\n=== Unique Device Values ===')
print(df_raw['Device'].unique())

In [ ]:
df = df_raw.copy()

# ── Monthly_Income ────────────────────────────────────────────────────
def clean_income(val):
    """Extract numeric value; return NaN for garbage."""
    if pd.isna(val):
        return np.nan
    val_str = str(val).replace(',', '').strip()
    # Remove text suffixes (Baht, THB, etc.)
    val_str = ''.join(c for c in val_str if c in '0123456789.e+E-')
    try:
        num = float(val_str)
    except ValueError:
        return np.nan
    # Flag: negative income → invalid
    if num < 0:
        return np.nan
    # Flag: unrealistic outlier (> 500,000 THB/month)
    if num > 500_000:
        return np.nan
    return num

df['Monthly_Income'] = df['Monthly_Income'].apply(clean_income)

# ── Time_on_App_min ───────────────────────────────────────────────────
def clean_time(val):
    if pd.isna(val):
        return np.nan
    try:
        num = float(val)
    except (ValueError, TypeError):
        return np.nan
    if num < 0 or num > 600:   # > 10 ชั่วโมง/ครั้ง → outlier
        return np.nan
    return num

df['Time_on_App_min'] = df['Time_on_App_min'].apply(clean_time)

# ── Last_Purchase_Days ────────────────────────────────────────────────
def clean_days(val):
    if pd.isna(val):
        return np.nan
    try:
        num = float(str(val).strip())
    except ValueError:
        return np.nan
    if num < 0:
        return np.nan
    return num

df['Last_Purchase_Days'] = df['Last_Purchase_Days'].apply(clean_days)

# ── Device ───────────────────────────────────────────────────────────
def clean_device(val):
    if pd.isna(val):
        return np.nan
    val_clean = str(val).replace('"', '').strip().lower()
    mapping = {'android': 'Android', 'ios': 'iOS', 'window': 'Windows', 'windows': 'Windows'}
    return mapping.get(val_clean, np.nan)

df['Device'] = df['Device'].apply(clean_device)

print('After cleaning — Missing Values:')
print(df[['Monthly_Income','Time_on_App_min','Last_Purchase_Days','Device']].isnull().sum())
print('\nDevice unique:', df['Device'].unique())

In [ ]:
# ── Impute Missing Values ─────────────────────────────────────────────
# ใช้ Median สำหรับ numeric (robust ต่อ outlier)
for col in ['Monthly_Income', 'Time_on_App_min', 'Last_Purchase_Days']:
    median_val = df[col].median()
    df[col].fillna(median_val, inplace=True)
    print(f'{col}: imputed with median = {median_val:.2f}')

# ใช้ Mode สำหรับ Device
mode_device = df['Device'].mode()[0]
df['Device'].fillna(mode_device, inplace=True)
print(f'Device: imputed with mode = {mode_device}')

print('\nMissing after imputation:', df.isnull().sum().sum())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Feature Distributions', fontsize=15, fontweight='bold')

num_cols = ['Monthly_Income', 'Time_on_App_min', 'Last_Purchase_Days']
colors = ['#4C72B0', '#DD8452', '#55A868']

for i, col in enumerate(num_cols):
    # Histogram by class
    for label, color in zip(['Yes', 'No'], ['#2ecc71', '#e74c3c']):
        axes[0, i].hist(df[df['Will_Buy_Again'] == label][col],
                        bins=20, alpha=0.6, label=label, color=color)
    axes[0, i].set_title(col)
    axes[0, i].legend()
    axes[0, i].set_xlabel(col)

# Device counts
device_counts = df.groupby(['Device', 'Will_Buy_Again']).size().unstack()
device_counts.plot(kind='bar', ax=axes[1, 0], color=['#e74c3c', '#2ecc71'], edgecolor='white')
axes[1, 0].set_title('Device vs Will_Buy_Again')
axes[1, 0].set_xlabel('')
axes[1, 0].tick_params(axis='x', rotation=0)

# Target distribution
target_counts = df['Will_Buy_Again'].value_counts()
axes[1, 1].pie(target_counts, labels=target_counts.index,
               autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
               startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1, 1].set_title('Target Distribution')

# Correlation heatmap
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            ax=axes[1, 2], center=0, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
axes[1, 2].set_title('Correlation Matrix')

plt.tight_layout()
plt.show()

In [ ]:
# ── One-Hot Encode Device ─────────────────────────────────────────────
df_encoded = pd.get_dummies(df.drop('Cust_ID', axis=1), columns=['Device'], drop_first=False)

# ── Encode Target ─────────────────────────────────────────────────────
df_encoded['Will_Buy_Again'] = (df_encoded['Will_Buy_Again'] == 'Yes').astype(int)

print('Features after encoding:')
print(df_encoded.columns.tolist())
print('\nShape:', df_encoded.shape)
df_encoded.head()

In [ ]:
# ── Train / Test Split ────────────────────────────────────────────────
feature_cols = [c for c in df_encoded.columns if c != 'Will_Buy_Again']
X = df_encoded[feature_cols].values
y = df_encoded['Will_Buy_Again'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── Standard Scaling ─────────────────────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train size: {X_train_sc.shape[0]}  |  Test size: {X_test_sc.shape[0]}')
print(f'Features  : {X_train_sc.shape[1]}')
print(f'Class balance (train) — 0:{(y_train==0).sum()}  1:{(y_train==1).sum()}')

In [ ]:
def build_model(input_dim, dropout_rate=0.3, learning_rate=1e-3):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),

        # Hidden layer 1
        layers.Dense(64, kernel_initializer='he_normal'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(dropout_rate),

        # Hidden layer 2
        layers.Dense(32, kernel_initializer='he_normal'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(dropout_rate),

        # Hidden layer 3
        layers.Dense(16, activation='relu'),

        # Output
        layers.Dense(1, activation='sigmoid')
    ], name='CustomerPurchaseNN')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 keras.metrics.AUC(name='auc'),
                 keras.metrics.Precision(name='precision'),
                 keras.metrics.Recall(name='recall')]
    )
    return model

model = build_model(input_dim=X_train_sc.shape[1])
model.summary()

In [ ]:
# ── Callbacks ─────────────────────────────────────────────────────────
early_stop = callbacks.EarlyStopping(
    monitor='val_auc', patience=15, restore_best_weights=True, mode='max'
)
lr_scheduler = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=8, min_lr=1e-6, verbose=1
)

# ── Train ─────────────────────────────────────────────────────────────
history = model.fit(
    X_train_sc, y_train,
    epochs=150,
    batch_size=16,
    validation_split=0.15,
    callbacks=[early_stop, lr_scheduler],
    verbose=1
)
print('\nTraining complete!')

In [ ]:
# ── Training Curves ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training History', fontsize=14, fontweight='bold')

metrics_to_plot = [('loss', 'Loss'), ('accuracy', 'Accuracy'), ('auc', 'AUC')]

for ax, (metric, title) in zip(axes, metrics_to_plot):
    ax.plot(history.history[metric], label='Train', linewidth=2, color='#3498db')
    ax.plot(history.history[f'val_{metric}'], label='Validation', linewidth=2,
            color='#e74c3c', linestyle='--')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Test Set Evaluation ───────────────────────────────────────────────
y_prob = model.predict(X_test_sc).flatten()
y_pred = (y_prob >= 0.5).astype(int)

print('='*50)
print('TEST SET RESULTS')
print('='*50)

results = model.evaluate(X_test_sc, y_test, verbose=0)
metric_names = ['Loss', 'Accuracy', 'AUC', 'Precision', 'Recall']
for name, val in zip(metric_names, results):
    print(f'  {name:12s}: {val:.4f}')

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Will NOT Buy', 'Will Buy']))

In [ ]:
# ── Confusion Matrix + ROC Curve ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Will NOT Buy', 'Will Buy'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix', fontweight='bold', fontsize=13)

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)
axes[1].plot(fpr, tpr, color='#3498db', linewidth=2.5,
             label=f'Neural Network (AUC = {auc_score:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random Classifier')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#3498db')
axes[1].set_xlabel('False Positive Rate', fontsize=11)
axes[1].set_ylabel('True Positive Rate', fontsize=11)
axes[1].set_title('ROC Curve', fontweight='bold', fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Probability Distribution ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))

ax.hist(y_prob[y_test == 0], bins=20, alpha=0.6, color='#e74c3c',
        label='Actual: Will NOT Buy', density=True)
ax.hist(y_prob[y_test == 1], bins=20, alpha=0.6, color='#2ecc71',
        label='Actual: Will Buy', density=True)
ax.axvline(0.5, color='black', linestyle='--', linewidth=1.5, label='Threshold = 0.5')
ax.set_xlabel('Predicted Probability', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Prediction Probability Distribution', fontweight='bold', fontsize=13)
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# สร้าง DataFrame ของ feature columns (ต้องตรงกับ training)
print('Feature columns used during training:')
print(feature_cols)

# ตัวอย่างลูกค้าใหม่ 3 คน
new_customers = pd.DataFrame({
    'Monthly_Income':    [35000, 8000, 70000],
    'Time_on_App_min':   [45.0,  5.0,  120.0],
    'Last_Purchase_Days':[7,     90,   3],
    'Device_Android':    [1, 0, 0],
    'Device_iOS':        [0, 1, 1],
    'Device_Windows':    [0, 0, 0],
})

# เติม columns ที่อาจขาด (ถ้า get_dummies สร้างไว้)
for col in feature_cols:
    if col not in new_customers.columns:
        new_customers[col] = 0
new_customers = new_customers[feature_cols]

new_scaled = scaler.transform(new_customers)
new_probs  = model.predict(new_scaled).flatten()
new_preds  = ['Will Buy 🟢' if p >= 0.5 else 'Will NOT Buy 🔴' for p in new_probs]

result_df = new_customers.copy()
result_df['Predicted_Prob'] = new_probs.round(4)
result_df['Prediction']     = new_preds

print('\n🔮 New Customer Predictions:')
result_df

In [ ]:
model.save('models/customer_purchase_nn.keras')
print('✅ Model saved as customer_purchase_nn.keras')